In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# IBM Circuit function

> **Note:** * Qiskit Functions jsou experimentální funkce dostupné pouze uživatelům IBM Quantum&reg; Premium Plan, Flex Plan a On-Prem (prostřednictvím IBM Quantum Platform API) Plan. Jsou ve stavu náhledového vydání a mohou se změnit.

## Přehled
IBM&reg; Circuit function přijímá [abstraktní PUBs](./primitive-input-output) jako vstupy a vrací zmírněné střední hodnoty jako výstupy. Tato Circuit function zahrnuje automatizovaný a přizpůsobený pipeline, který umožňuje výzkumníkům soustředit se na objevování algoritmů a aplikací.
## Popis
Po odeslání svého PUB jsou tvé abstraktní obvody a observabely automaticky transpilovány, spuštěny na hardwaru a post-processovány, aby vrátily zmírněné střední hodnoty. K tomu kombinuje následující nástroje:

- [Qiskit Transpiler Service](./qiskit-transpiler-service), včetně automatického výběru transpilačních průchodů řízených umělou inteligencí a heuristickými metodami pro překlad tvých abstraktních obvodů na ISA obvody optimalizované pro hardware
- [Potlačení a zmírnění chyb potřebné pro výpočty v užitkovém měřítku](./error-mitigation-and-suppression-techniques), včetně měřicího a hradlového twirling, dynamického oddělování, Twirled Readout Error eXtinction (TREX), Zero-Noise Extrapolation (ZNE) a Probabilistic Error Amplification (PEA)
- [Qiskit Runtime Estimator](./get-started-with-primitives), pro spouštění ISA PUBs na hardwaru a vracení zmírněných středních hodnot

![IBM Circuit function](../docs/images/guides/ibm-circuit-function/ibm-circuit-function.svg)
## Začínáme
Autentizuj se pomocí svého [API klíče](http://quantum.cloud.ibm.com/) a vyber Qiskit Function takto. (Tento úryvek předpokládá, že jsi již [uložil svůj účet](/guides/functions#install-qiskit-functions-catalog-client) do svého místního prostředí.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("ibm/circuit-function")

## Příklad
 Začni tímto základním příkladem:

In [2]:
from qiskit.circuit.random import random_circuit
from qiskit_ibm_runtime import QiskitRuntimeService

# You can skip this step if you have a target backend, e.g.
# backend_name = "ibm_brisbane"
# You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits
pubs = [(circuit, observable)]

job = function.run(
    # Use `backend_name=backend_name` if you didn't initialize a backend object
    backend_name=backend.name,
    pubs=pubs,
)

Zkontroluj [stav](/guides/functions#check-job-status) úlohy svého Qiskit Function nebo získej [výsledky](/guides/functions#retrieve-results) takto:

In [3]:
print(job.status())
result = job.result()

QUEUED


The results have the same format as an [Estimator result](/docs/guides/primitive-input-output#estimator-output):

In [4]:
print(f"The result of the submitted job had {len(result)} PUB\n")
print(
    f"The associated PubResult of this job has the following DataBins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB

The associated PubResult of this job has the following DataBins:
 DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>))

And this DataBin has attributes: dict_keys(['evs', 'stds', 'ensemble_standard_error'])
The expectation values measured from this PUB are: 
1.02116704805492


Výsledky mají stejný formát jako [výsledek Estimatoru](/guides/primitive-input-output#estimator-output):

In [5]:
options = {"mitigation_level": 2}

job = function.run(backend_name=backend.name, pubs=pubs, options=options)

#### All available options

In addition to `mitigation_level`, the IBM Circuit function also provides a select number of advanced options that allow you to fine-tune the cost/accuracy trade-off. The following table shows all the available options:

| Option | Sub-option | Sub-sub-option | Description | Choices | Default |
|-|-|-|-|-|-|
| default_precision |  |  | The default precision to use for any PUB or `run()`<br/>call that does not specify one.<br/>Each input PUB can specify its own precision. If the `run()` method is given a precision, then that value is used for all PUBs in the `run()` call that do not specify their own.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Maximum execution time in seconds, which is based<br/>on QPU usage (not wall clock time). QPU usage is the<br/>amount of time that the QPU is dedicated to processing your job. If a job exceeds this time limit, it is forcibly canceled. | Integer number of seconds in the range [1, 10800] |  |
| mitigation_level |  |  | How much error suppression and mitigation to apply. Refer to the [Mitigation level](#mitigation-level) section for more information about the methods used at each level. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | How much optimization to perform on the circuits. [Higher levels](/docs/guides/set-optimization) generate more optimized circuits, at the expense of longer transpilation time. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Whether to enable dynamical decoupling. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) for the explanation of the method.  | True/False | True |
|  | sequence_type |  | Which dynamical decoupling sequence to use.<br/>* `XX`: use the sequence `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: use the sequence `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: use the sequence<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Whether to apply 2-qubit Clifford gate twirling. | True/False | False |
|  | enable_measure |  | Whether to enable twirling of measurements. | True/False | True |
| resilience | measure_mitigation |  | Whether to enable TREX measurement error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) for the explanation of the method.  | True/False | True |
|  | zne_mitigation |  | Whether to turn on Zero Noise Extrapolation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) for the explanation of the method.  | True/False | False |
|  | zne | amplifier | Which technique to use for amplifying noise. One of: <br/> - `gate_folding` (default) uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are chosen randomly.<br/><br/> - `gate_folding_front` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the front of the topologically ordered DAG circuit.<br/><br/> - `gate_folding_back` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the back of the topologically ordered DAG circuit.<br/><br/> - `pea` uses a technique called Probabilistic error amplification (PEA) to amplify noise. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) for the explanation of the method.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Noise factors to use for noise amplification. | list of floats; each float >= 1 | (1, 1.5, 2) for PEA, and (1, 3, 5) otherwise. |
|  |  | extrapolator | Noise factors to evaluate the fit extrapolation models at. This option does not affect execution or model fitting in any way; it only determines the points at which the `extrapolator` objects are evaluated to be returned in the data fields called `evs_extrapolated` and `stds_extrapolated`. | one or more of `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Whether to turn on Probabilistic Error Cancellation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) for the explanation of the method.  | True/False | False |
|  | pec | max_overhead | The maximum circuit sampling overhead allowed, or `None` for no maximum. | None/ integer >1 | 100 |

In the following example, setting the mitigation level to 1 initially turns off ZNE mitigation, but setting `zne_mitigation` to `True` overrides the relevant setup from `mitigation_level`.

In [6]:
options = {"mitigation_level": 1, "resilience": {"zne_mitigation": True}}

## Vstupy
Viz následující tabulku pro všechny vstupní parametry, které IBM Circuit function přijímá. Následující sekce [Možnosti](#options) obsahuje další podrobnosti o dostupných `options`.
| Název      | Typ                       | Popis                                                                                                                                                                                                                         | Povinné | Výchozí                                                                  | Příklad                                  |
|-----------|----------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|------------------------------------------|
| backend_name   | str                        | Název Backend, na který se má dotaz odeslat.                                                                                                                                                                                              | Ano      | Není                                                                      | `ibm_fez`                                |
| pubs      | Iterable[EstimatorPubLike] | Iterovatelná kolekce abstraktních objektů podobných PUB (primitive unified bloc), jako jsou n-tice `(circuit, observables)` nebo `(circuit, observables, parameter_values)`. Viz [Přehled PUBs](/guides/primitive-input-output#overview-of-pubs) pro více informací. Obvody mohou být abstraktní (non-ISA). | Ano      | Není                                                                      | (circuit, observables, parameter_values) |
| options   | dict                       | Vstupní možnosti. Viz sekci **Možnosti** pro více podrobností.                                                                                                                                                                                | Ne       | Viz sekci **Možnosti** pro podrobnosti.                                                   | `{"optimization_level": 3}`                |
| instance  | str                        | Název cloudového prostředku instance, která se má použít ve daném formátu.                                                                                                                                                                                        | Ne       | Jedna je náhodně vybrána, pokud má tvůj účet přístup k více instancím. | `CRN`                   |
### Možnosti
#### Struktura
Podobně jako u Qiskit Runtime primitiv lze možnosti pro IBM Circuit function specifikovat jako vnořený slovník. Běžně používané možnosti, jako například ``optimization_level`` a ``mitigation_level``, jsou na první úrovni. Ostatní, pokročilejší možnosti jsou seskupeny do různých kategorií, jako například ``resilience``.

#### Výchozí hodnoty
Pokud neuvedeš hodnotu pro určitou možnost, použije se výchozí hodnota specifikovaná službou.

#### Úroveň zmírnění
IBM Circuit function také podporuje `mitigation_level`. Úroveň zmírnění určuje, kolik potlačení a zmírnění chyb se má na úlohu aplikovat. Vyšší úrovně generují přesnější výsledky, za cenu delší doby zpracování. Míra snížení chyb závisí na použité metodě. Úroveň zmírnění abstrahuje podrobnou volbu metod zmírnění a potlačení chyb, aby uživatelé mohli uvažovat o kompromisu cena/přesnost vhodném pro jejich aplikaci. Následující tabulka zobrazuje odpovídající metody pro každou úroveň.

> **Note:** Přestože jsou názvy podobné, `mitigation_level` aplikuje jiné techniky než ty, které používá `resilience_level` Estimatoru.

Podobně jako ``resilience_level`` v Qiskit Runtime Estimatoru specifikuje ``mitigation_level`` základní sadu vybraných možností. Veškeré možnosti, které ručně specifikuješ nad rámec úrovně zmírnění, se aplikují na vrchol základní sady možností definované touto úrovní. V principu tedy můžeš nastavit úroveň zmírnění na 1 a zároveň vypnout zmírnění měření, přestože to není doporučeno.

| **Úroveň zmírnění** | **Technika** |
|:-:|:-:|
| 1 [Výchozí] | Dynamické oddělování + měřicí twirling + TREX  |
| 2 | Úroveň 1 + hradlový twirling + ZNE pomocí hradlového skládání |
| 3 | Úroveň 1 + hradlový twirling + ZNE pomocí PEA |

Následující příklad demonstruje nastavení úrovně zmírnění:

In [7]:
print(f"The result of the submitted job had {len(result)} PUB")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)
print(f"And the associated metadata is: \n{result[0].metadata}")

The result of the submitted job had 1 PUB
The expectation values measured from this PUB are: 
1.02116704805492
And the associated metadata is: 
{'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


#### Všechny dostupné možnosti
Kromě `mitigation_level` poskytuje IBM Circuit function také omezený počet pokročilých možností, které ti umožňují doladit kompromis cena/přesnost. Následující tabulka zobrazuje všechny dostupné možnosti:

| Možnost | Podmožnost | Podmožnost podmožnosti | Popis | Volby | Výchozí |
|-|-|-|-|-|-|
| default_precision |  |  | Výchozí přesnost, která se použije pro jakýkoli PUB nebo volání `run()`<br/>která nespecifikuje vlastní.<br/>Každý vstupní PUB může specifikovat svou vlastní přesnost. Pokud je metodě `run()` zadána přesnost, použije se tato hodnota pro všechny PUBs ve volání `run()`, které nespecifikují vlastní.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Maximální doba provádění v sekundách, která je založena<br/>na využití QPU (nikoli na skutečném čase). Využití QPU je<br/>množství času, po které je QPU věnována zpracování tvé úlohy. Pokud úloha překročí tento časový limit, je nuceně zrušena. | Celé číslo sekund v rozsahu [1, 10800] |  |
| mitigation_level |  |  | Kolik potlačení a zmírnění chyb aplikovat. Viz sekci [Úroveň zmírnění](#mitigation-level) pro více informací o metodách používaných na každé úrovni. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | Kolik optimalizace provést na obvodech. [Vyšší úrovně](/guides/set-optimization) generují optimalizovanější obvody, za cenu delší doby transpilace. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Zda povolit dynamické oddělování. Viz [Techniky potlačení a zmírnění chyb](/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) pro vysvětlení metody.  | True/False | True |
|  | sequence_type |  | Která sekvence dynamického oddělování se má použít.<br/>* `XX`: použije sekvenci `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: použije sekvenci `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: použije sekvenci<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Zda aplikovat twirling 2-Qubit Cliffordových Gate. | True/False | False |
|  | enable_measure |  | Zda povolit twirling měření. | True/False | True |
| resilience | measure_mitigation |  | Zda povolit metodu zmírnění chyb měření TREX. Viz [Techniky potlačení a zmírnění chyb](/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) pro vysvětlení metody.  | True/False | True |
|  | zne_mitigation |  | Zda zapnout metodu zmírnění chyb Zero Noise Extrapolation. Viz [Techniky potlačení a zmírnění chyb](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) pro vysvětlení metody.  | True/False | False |
|  | zne | amplifier | Která technika se má použít pro amplifikaci šumu. Jedna z: <br/> - `gate_folding` (výchozí) používá 2-Qubit gate folding pro amplifikaci šumu. Pokud faktor šumu vyžaduje amplifikaci pouze podmnožiny Gate, jsou tyto Gate vybrány náhodně.<br/><br/> - `gate_folding_front` používá 2-Qubit gate folding pro amplifikaci šumu. Pokud faktor šumu vyžaduje amplifikaci pouze podmnožiny Gate, jsou tyto Gate vybrány z přední části topologicky uspořádaného DAG Circuit.<br/><br/> - `gate_folding_back` používá 2-Qubit gate folding pro amplifikaci šumu. Pokud faktor šumu vyžaduje amplifikaci pouze podmnožiny Gate, jsou tyto Gate vybrány ze zadní části topologicky uspořádaného DAG Circuit.<br/><br/> - `pea` používá techniku zvanou Probabilistic error amplification (PEA) pro amplifikaci šumu. Viz [Techniky potlačení a zmírnění chyb](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) pro vysvětlení metody.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Faktory šumu, které se použijí pro amplifikaci šumu. | seznam floatů; každý float >= 1 | (1, 1.5, 2) pro PEA a (1, 3, 5) jinak. |
|  |  | extrapolator | Faktory šumu, při nichž se vyhodnocují modely extrapolace fitování. Tato možnost nijak neovlivňuje provádění ani fitování modelu; pouze určuje body, ve kterých jsou objekty `extrapolator` vyhodnoceny a vráceny v datových polích `evs_extrapolated` a `stds_extrapolated`. | jeden nebo více z `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Zda zapnout metodu zmírnění chyb Probabilistic Error Cancellation. Viz [Techniky potlačení a zmírnění chyb](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) pro vysvětlení metody.  | True/False | False |
|  | pec | max_overhead | Maximální povolená režie vzorkování obvodů, nebo `None` pro bez maxima. | None/ celé číslo >1 | 100 |

V následujícím příkladu nastavení úrovně zmírnění na 1 nejprve vypne ZNE zmírnění, ale nastavení `zne_mitigation` na `True` přepíše příslušné nastavení z `mitigation_level`.

In [8]:
job = function.run(
    backend_name="bad_backend_name", pubs=pubs, options=options
)

print(job.result())

QiskitServerlessException: "Traceback (most recent call last):\n  File \"/runner/runner.py\", line 10, in run\n    func = CircuitFunction(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/runner/circuit_function/circuit_function.py\", line 87, in __init__\n    self._backend = self._service.backend(\n                    ^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 754, in backend\n    backends = self.backends(name, instance=instance, use_fractional_gates=use_fractional_gates)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 497, in backends\n    raise QiskitBackendNotFoundError(\"No backend matches the criteria.\")\nqiskit.providers.exceptions.QiskitBackendNotFoundError: 'No backend matches the criteria.'\n"

## Výstupy
Výstupem Circuit function je [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), který obsahuje dvě pole:

- Jeden nebo více objektů [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult). Ty lze indexovat přímo z `PrimitiveResult`.

- Metadata na úrovni úlohy.

Každý `PubResult` obsahuje pole `data` a `metadata`.

- Pole `data` obsahuje alespoň pole středních hodnot (`PubResult.data.evs`) a pole standardních chyb (`PubResult.data.stds`). Může také obsahovat další data v závislosti na použitých možnostech.

- Pole `metadata` obsahuje metadata na úrovni PUB (`PubResult.metadata`).

Následující úryvek kódu popisuje formát `PrimitiveResult` (a přidruženého `PubResult`).